In [21]:
from dotenv import load_dotenv
from lmstudio import get_lmstudio_client

load_dotenv()

openai_client = get_lmstudio_client()
MODEL = "llama3.1:8b"

In [ ]:
def llm(prompt):
    response = openai_client.responses.create(
        model=MODEL,
        input=prompt,
    )
    return response.output_text

In [8]:
llm("what is the capital of France?")

'The capital of France is Paris!'

In [9]:
question = "I just discovered the course. Can I join now?"

In [10]:
from ingest import load_faq_data, build_index
index = build_index(load_faq_data())


In [11]:
INSTRUCTIONS = f'''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

'''

In [12]:
def search_index(question):
    return index.search(question, 
    boost_dict={"question": 3.0, "answer": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},

    num_results=5)

In [13]:
def build_prompt(question, search_results):
    return f'''
    {INSTRUCTIONS}

    
    Question: {question}

    Context:
    {search_results}
    '''

In [14]:
def build_context(search_results):

    lines = []
    for doc in search_results:
        lines.append(doc['section'])
        lines.append("Q: " + doc['question'])
        lines.append("A: " + doc['answer'])
        lines.append("")
    return '\n'.join(lines).strip()



In [15]:
def rag(question):
    search_results = search_index(question)
    user_prompt = build_prompt(question, build_context(search_results))    
    return llm(user_prompt)

In [16]:
res = rag(question)

In [17]:
res

'Yes, you can join now! According to the context, "Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions." This implies that there is no time limit for joining the course, but if you want to receive a certificate, you need to submit your project during the specified timeframe.'

In [18]:
context = build_context(search_index(question))
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is run

In [19]:
prompt = build_prompt(question, context)

In [23]:
response = openai_client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}],
)


In [24]:
response.choices[0].message.content

'Yes, you can join now. However, if you want to receive a certificate, you should submit your project before the deadline to be eligible.'

In [25]:
response.choices[0].message.content

'Yes, you can join now. However, if you want to receive a certificate, you should submit your project before the deadline to be eligible.'

In [26]:
response.usage

CompletionUsage(completion_tokens=30, prompt_tokens=395, total_tokens=425, completion_tokens_details=None, prompt_tokens_details=None)

In [27]:
def llm(instructions, user_prompt, model=MODEL):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt},
    ]

    response = openai_client.chat.completions.create(
        model=model,
        messages=message_history,
    )
    return response.choices[0].message.content


In [28]:
def rag(query, model=MODEL):
    search_results = search_index(query)
    prompt = build_prompt(query, build_context(search_results))
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer


In [29]:
answer = rag(question)
print(answer)

Yes, you can join now. If you want to receive a certificate, make sure to submit your project while submissions are still accepted.


In [30]:
from rag_helper import RAGBase

rag_helper = RAGBase(
    index=index,
    llm_client=openai_client,
    model=MODEL,
)

In [31]:
rag_helper.rag(question)

"**Yes, you can join now**.\n\nFrom the provided context, we know that:\n\n* You don't need to register before joining; registration is just to gauge interest.\n* If you want a certificate, you need to submit your project while submissions are still open.\n* You can start learning and submitting homework at any time.\n\nHowever, keep in mind that if you want to receive a certificate, you need to follow the live cohort and peer-review projects when the course is running."